# 5. Análise dos resultados e exportação para o Power BI
Este notebook resume a classificação pelos eixos da BNCC e organiza os dados em tabelas que podem ser importadas no Power BI.

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.table import Table, TableStyleInfo

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')

pasta_processados = raiz / 'dados' / '1_processados'
pasta_consumo = raiz / 'dados' / '2_consumo'
pasta_consumo.mkdir(parents=True, exist_ok=True)

## 5.1 Leitura dos resultados da classificação

In [ ]:
artigos = pd.read_csv(pasta_processados / '04_artigos_classificados_bncc.csv', encoding='utf-8-sig')
classificacoes = pd.read_csv(pasta_processados / '04_classificacoes_bncc.csv', encoding='utf-8-sig')
termos_titulos = pd.read_csv(pasta_processados / '03_termos_titulos.csv', encoding='utf-8-sig')
bigramas_titulos = pd.read_csv(pasta_processados / '03_bigramas_titulos.csv', encoding='utf-8-sig')
ranking_termos = pd.read_csv(pasta_processados / '03_ranking_termos_titulos.csv', encoding='utf-8-sig')
ranking_bigramas = pd.read_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', encoding='utf-8-sig')
freq_termos_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_termos_ano_evento.csv', encoding='utf-8-sig')
freq_bigramas_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_bigramas_ano_evento.csv', encoding='utf-8-sig')

artigos['ano'] = pd.to_numeric(artigos['ano'], errors='coerce').astype('Int64')

print(f'Artigos analisados: {len(artigos)}')
print(f'Relações artigo–eixo: {len(classificacoes)}')

## 5.6 Preparação do modelo de dados

In [ ]:
tabela_artigo_bncc = classificacoes[['id_artigo', 'eixo_bncc']].copy()
tabela_artigo_bncc['eixo_bncc'] = tabela_artigo_bncc['eixo_bncc'].astype(str).str.strip()
tabela_artigo_bncc = (
    tabela_artigo_bncc[tabela_artigo_bncc['eixo_bncc'] != '']
    .drop_duplicates()
    .reset_index(drop=True)
)

freq_bncc_ano_evento = (
    tabela_artigo_bncc
    .merge(artigos[['id_artigo', 'ano', 'evento']], on='id_artigo', how='left')
    .groupby(['ano', 'evento', 'eixo_bncc'])
    .size()
    .reset_index(name='frequencia')
)

print(f'Linhas tabela BNCC: {len(tabela_artigo_bncc)}')
print(f'Frequências BNCC por ano/evento: {len(freq_bncc_ano_evento)}')

## 5.7 Exportação dos arquivos XLSX

In [ ]:
def formatar_workbook(writer):
    for indice, planilha in enumerate(writer.book.worksheets, start=1):
        planilha.freeze_panes = 'A2'
        planilha.auto_filter.ref = planilha.dimensions
        for celula in planilha[1]:
            celula.fill = PatternFill('solid', fgColor='1F4E78')
            celula.font = Font(color='FFFFFF', bold=True)
            celula.alignment = Alignment(horizontal='center')
        for coluna in planilha.columns:
            valores = [str(c.value) if c.value is not None else '' for c in coluna[:200]]
            largura = min(max(max((len(v) for v in valores), default=0) + 2, 12), 60)
            planilha.column_dimensions[coluna[0].column_letter].width = largura
        if planilha.max_row > 1 and planilha.max_column > 0:
            tabela = Table(displayName=f'Tabela{indice}', ref=planilha.dimensions)
            tabela.tableStyleInfo = TableStyleInfo(name='TableStyleMedium2', showRowStripes=True)
            planilha.add_table(tabela)


def criar_identificador_unico(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
    df = df.copy()
    if 'evento' in df.columns and coluna in df.columns:
        df[f'{coluna}_original'] = df[coluna].astype(str).str.strip()
        df[coluna] = (
            df['evento'].fillna('').astype(str).str.strip()
            + ' | '
            + df[coluna].fillna('').astype(str).str.strip()
        )
    return df

termos_titulos = criar_identificador_unico(termos_titulos, 'termo')
bigramas_titulos = criar_identificador_unico(bigramas_titulos, 'bigrama')
ranking_termos = criar_identificador_unico(ranking_termos, 'termo')
ranking_bigramas = criar_identificador_unico(ranking_bigramas, 'bigrama')
freq_termos_ano_evento = criar_identificador_unico(freq_termos_ano_evento, 'termo')
freq_bigramas_ano_evento = criar_identificador_unico(freq_bigramas_ano_evento, 'bigrama')

arquivo_powerbi = pasta_consumo / '05_powerbi_bncc.xlsx'
with pd.ExcelWriter(arquivo_powerbi, engine='openpyxl') as writer:
    artigos.to_excel(writer, sheet_name='Artigos', index=False)
    termos_titulos.to_excel(writer, sheet_name='Termos', index=False)
    bigramas_titulos.to_excel(writer, sheet_name='Bigramas', index=False)
    ranking_termos.to_excel(writer, sheet_name='Ranking_Termos_Relevantes', index=False)
    ranking_bigramas.to_excel(writer, sheet_name='Ranking_Bigramas_Relevantes', index=False)
    freq_termos_ano_evento.to_excel(writer, sheet_name='Freq_Termos_Ano_Evento', index=False)
    freq_bigramas_ano_evento.to_excel(writer, sheet_name='Freq_Bigramas_Ano_Evento', index=False)
    tabela_artigo_bncc.to_excel(writer, sheet_name='Artigo_BNCC', index=False)
    freq_bncc_ano_evento.to_excel(writer, sheet_name='Freq_BNCC_Ano_Evento', index=False)
    formatar_workbook(writer)

print(f'Gerado: {arquivo_powerbi}')